In [1]:
# -*- coding: utf-8 -*-
# ADF: statsmodels.tsa.stattools.adfuller
# KPSS: statsmodels.tsa.stattools.kpss
# auto.arima(): pmdarima.arima.auto_arima
# ARIMA ręcznie: statsmodels.tsa.statespace.SARIMAX
# Diagnostyka: Ljung-Box, Shapiro, ACF/PACF reszt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import shapiro
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.stattools import acf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
import statsmodels.api as sm
from pmdarima import auto_arima
from pmdarima.metrics import smape

np.random.seed(123)

###############################################
# 1. Działanie testów ADF i KPSS na różnych symulowanych szeregach
###############################################

N = 300
t = np.arange(1, N + 1)
a0 = 2.0
a1 = 0.1

# gaussowski biały szum {e(t)}
epst = np.random.normal(size=N)

# Symulacja modeli {X(t)}
Xt1 = np.cumsum(epst)                      # błądzenie losowe bez dryfu - I(1)
Xt2 = np.cumsum(epst + a0)                 # błądzenie losowe z dryfem - I(1) z trendem deterministycznym
Xt3 = a1 * t + a0 + epst                   # trendostacjonarny (liniowy trend + brak unit root)
# AR(1) stacjonarny
phi = 0.8
Xt4 = sm.tsa.ArmaProcess(ar=[1, -phi], ma=[1]).generate_sample(nsample=N, burnin=200)

# ARIMA(2,1,1) niestacjonarny (z unit root)
arparams = np.array([0.4, -0.25])
maparams = np.array([0.5])
# wygenerujemy ARIMA(2,1,1) przez zróżnicowanie odwrotne na ARMA(2,1)
eps = np.random.normal(size=N)
arma = sm.tsa.ArmaProcess(ar=np.r_[1, -arparams], ma=np.r_[1, maparams]).generate_sample(nsample=N, burnin=300)
Xt5 = np.cumsum(arma)  # integrowanie (d=1) → ARIMA(2,1,1)

# Wizualizacja
plt.figure(figsize=(12, 8))
plt.subplot(2, 2, 1); plt.plot(Xt1); plt.title("Trajektoria szeregu {X1(t)}"); plt.xlabel("t"); plt.grid(True)
plt.subplot(2, 2, 2); plt.plot(Xt2); plt.title("Trajektoria szeregu {X2(t)}"); plt.xlabel("t"); plt.grid(True)
plt.subplot(2, 2, 3); plt.plot(Xt3); plt.title("Trajektoria szeregu {X3(t)}"); plt.xlabel("t"); plt.grid(True)
plt.subplot(2, 2, 4); plt.plot(Xt4); plt.title("Trajektoria szeregu {X4(t)}"); plt.xlabel("t"); plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 3.5))
plt.plot(Xt5)
plt.title("Trajektoria szeregu {X(t)} ~ ARIMA(2,1,1)")
plt.xlabel("t"); plt.grid(True)
plt.tight_layout()
plt.show()

def adf_test(x):
    res = adfuller(x, autolag='AIC')  # stat, pvalue, usedlag, nobs, crit, icbest
    return {"stat": res[0], "pvalue": res[1]}

def kpss_test(x, regression='c'):
    # regression='c' (poziom), 'ct' (trend) – analog R: null="Level"/"Trend"
    stat, pvalue, lags, crit = kpss(x, regression=regression, nlags='auto')
    return {"stat": stat, "pvalue": pvalue}

# Testy (uwaga: dla trendostacjonarnego KPSS użyjemy "ct" = trend)
TestDF1, TestKP1 = adf_test(Xt1), kpss_test(Xt1, regression='c')
TestDF2, TestKP2 = adf_test(Xt2), kpss_test(Xt2, regression='c')
TestDF3, TestKP3 = adf_test(Xt3), kpss_test(Xt3, regression='ct')
TestDF4, TestKP4 = adf_test(Xt4), kpss_test(Xt4, regression='c')
TestDF5, TestKP5 = adf_test(Xt5), kpss_test(Xt5, regression='c')

print("### Wyniki testów (statystyka, p-value):")
print("Xt1  ADF:", (TestDF1["stat"], TestDF1["pvalue"]), " | KPSS:", (TestKP1["stat"], TestKP1["pvalue"]))
print("Xt2  ADF:", (TestDF2["stat"], TestDF2["pvalue"]), " | KPSS:", (TestKP2["stat"], TestKP2["pvalue"]))
print("Xt3  ADF:", (TestDF3["stat"], TestDF3["pvalue"]), " | KPSS:", (TestKP3["stat"], TestKP3["pvalue"]))
print("Xt4  ADF:", (TestDF4["stat"], TestDF4["pvalue"]), " | KPSS:", (TestKP4["stat"], TestKP4["pvalue"]))
print("Xt5  ADF:", (TestDF5["stat"], TestDF5["pvalue"]), " | KPSS:", (TestKP5["stat"], TestKP5["pvalue"]))

# Interpretacja progowa (poziom 0.05):
# ADF: p < 0.05 → odrzucamy H0 (brak unit root)
# KPSS: p < 0.05 → odrzucamy H0 (nie jest stacjonarny)

###############################################
# 2. Symulacja mocy testów ADF i KPSS (Monte Carlo)
###############################################

N = 300
MC = 10_000   # Uwaga: duże obciążenie; do szybkiego testu ustaw np. 1000

def simulate_power_ar1(phi=0.95, N=300, MC=10000, alpha=0.05):
    rejections_adf = 0
    rejections_kpss = 0
    for _ in range(MC):
        x = sm.tsa.ArmaProcess(ar=[1, -phi], ma=[1]).generate_sample(nsample=N, burnin=200)
        # ADF
        if adfuller(x, autolag='AIC')[1] <= alpha:
            rejections_adf += 1
        # KPSS (poziom)
        if kpss(x, regression='c', nlags='auto')[1] <= alpha:
            rejections_kpss += 1
    power_adf = rejections_adf / MC
    power_kpss = rejections_kpss / MC
    return power_adf, power_kpss

power_adf, power_kpss = simulate_power_ar1(phi=0.95, N=N, MC=MC, alpha=0.05)
print("\nMoce testów (częstości odrzuceń H0):")
print("ADF  :", power_adf)
print("KPSS :", power_kpss)

###############################################
# 3. Studium analizy danych empirycznych
#    Dane: plik "HOUSEHOLDS.csv" (kolumny: Data, Liczba)
###############################################

# a) wczytanie danych i wstępna wizualizacja
# Oczekiwany format Data: YYYY-MM lub YYYY (dopasujemy parser)
def parse_date(s):
    try:
        return pd.to_datetime(s)
    except Exception:
        # awaryjnie – jeśli rok
        return pd.to_datetime(s, format="%Y")

dane = pd.read_csv("HOUSEHOLDS.csv")
dane.columns = ["Data", "Liczba"]
dane["Data"] = dane["Data"].apply(parse_date)
dane = dane.sort_values("Data").reset_index(drop=True)

Xt = dane["Liczba"].astype(float).values

# Założyłeś miesięczne dane → freq=12
# Jeśli masz roczne, zmień na 1.
freq = 12
start_year = dane.loc[0, "Data"].year
start_month = dane.loc[0, "Data"].month if freq == 12 else 1

# Tworzymy obiekt pandas Series z indeksem DatetimeIndex
Xts = pd.Series(Xt, index=dane["Data"])
Xts = Xts.asfreq("MS") if freq == 12 else Xts.asfreq("YS")  # wymuszamy równomierną siatkę

plt.figure(figsize=(10, 4))
plt.plot(Xts)
plt.title("Dane empiryczne: szacowana liczba gospodarstw domowych w USA [tys.]")
plt.xlabel("Time"); plt.ylabel("X(t)"); plt.grid(True)
plt.tight_layout()
plt.show()

# b) próbkowe ACF i testy pierwiastka jednostkowego
fig, ax = plt.subplots(1, 1, figsize=(8, 3.5))
plot_acf(Xts.dropna(), lags=40, ax=ax)
ax.set_title("Próbkowa ACF dla szeregu X(t)")
plt.tight_layout()
plt.show()

# ADF i KPSS
adf_res = adfuller(Xts.dropna(), autolag='AIC')
kpss_res = kpss(Xts.dropna(), regression='c', nlags='auto')
print("\nADF: stat=%.4f, p=%.4f" % (adf_res[0], adf_res[1]))
print("KPSS (poziom): stat=%.4f, p=%.4f" % (kpss_res[0], kpss_res[1]))

# c) automatyczne dopasowanie modelu ARIMA (pmdarima.auto_arima)
#   stepwise=True szybsze; seasonal automatycznie z freq=12
auto_model = auto_arima(
    Xts,
    seasonal=(freq == 12),
    m=freq if freq == 12 else 1,
    stepwise=True,
    trace=False,
    information_criterion="aic",
    suppress_warnings=True
)

print("\n=== auto.arima() odpowiednik ===")
print(auto_model.summary())

# ogólna diagnostyka (AIC/BIC/AICc – w pmdarima summary)
# współczynniki:
print("\nWspółczynniki modelu auto_arima:")
print(auto_model.params())

# dane empiryczne vs fitted
fitted = pd.Series(auto_model.predict_in_sample(), index=Xts.index)
comp = pd.DataFrame({"y": Xts, "fitted": fitted})
print("\nPodgląd (y, fitted):")
print(comp.tail())

# analiza reszt
resid = (Xts - fitted).dropna()

# Wykres reszt + ACF + QQ
plt.figure(figsize=(9, 6))
plt.subplot(3,1,1); plt.plot(resid); plt.title("Reszty modelu auto_arima"); plt.grid(True)
plt.subplot(3,1,2); plot_acf(resid, ax=plt.gca(), lags=40)
plt.subplot(3,1,3); sm.qqplot(resid, line="s", ax=plt.gca()); plt.title("QQ-plot reszt")
plt.tight_layout(); plt.show()

# Test normalności (Shapiro-Wilk)
sh = shapiro(resid.values)
print("Shapiro-Wilk (reszty): stat=%.4f, p=%.4f" % (sh.statistic, sh.pvalue))

# Ljung-Box na resztach (brak autokorelacji w resztach)
lb = acorr_ljungbox(resid, lags=[10, 20, 30], return_df=True)
print("\nLjung-Box p-values:\n", lb["lb_pvalue"])

# d) Prognozy ex-ante
H = 24
fct = auto_model.predict(n_periods=H, return_conf_int=True)
fct_mean = pd.Series(fct[0], index=pd.date_range(Xts.index[-1] + Xts.index.freq, periods=H, freq=Xts.index.freq))
fct_ci = pd.DataFrame(fct[1], index=fct_mean.index, columns=["lower", "upper"])

plt.figure(figsize=(10, 4))
plt.plot(Xts, label="y")
plt.plot(fct_mean, label="prognoza")
plt.fill_between(fct_mean.index, fct_ci["lower"], fct_ci["upper"], alpha=0.2, label="95% CI")
plt.title("Prognozy ex-ante dla szeregu {X(t)} z auto_arima()")
plt.xlabel("t"); plt.ylabel("X(t)"); plt.grid(True); plt.legend()
plt.tight_layout(); plt.show()

# "accuracy" – przykładowo SMAPE fitu w próbie:
print("SMAPE (fit in-sample):", smape(Xts.iloc[auto_model.arima_res_.nobs_:], fitted.iloc[auto_model.arima_res_.nobs_:]))

###############################################
# e) Ręczna estymacja ARIMA + diagnostyka (odpowiedniki Arima())
###############################################

# Uwaga: w R w sekcji b) jest podpis "ARIMA(1,1,1) ?" ale kod używa order=c(1,2,1).
# Poniżej pokazuję (1,0,1) i (1,2,1), jak w Twoim skrypcie.

H = 24

# a) ARMA(1,1): order=(1,0,1)
mod1 = sm.tsa.statespace.SARIMAX(Xts, order=(1,0,1), enforce_stationarity=False, enforce_invertibility=False)
res1 = mod1.fit(disp=False)
print("\n=== Model ręczny: ARMA(1,1) ===")
print(res1.summary())

resid1 = res1.resid.dropna()

# „checkresiduals” style
plt.figure(figsize=(9, 6))
plt.subplot(3,1,1); plt.plot(resid1); plt.title("Reszty ARMA(1,1)"); plt.grid(True)
plt.subplot(3,1,2); plot_acf(resid1, ax=plt.gca(), lags=40)
plt.subplot(3,1,3); sm.qqplot(resid1, line="s", ax=plt.gca()); plt.title("QQ-plot reszt")
plt.tight_layout(); plt.show()

print("Shapiro-Wilk (reszty ARMA(1,1)):", shapiro(resid1.values))
print("Ljung-Box p-values:\n", acorr_ljungbox(resid1, lags=[10, 20, 30], return_df=True)["lb_pvalue"])

fct1_res = res1.get_forecast(steps=H)
fct1_mean = fct1_res.predicted_mean
fct1_ci = fct1_res.conf_int()

plt.figure(figsize=(10, 4))
plt.plot(Xts, label="y")
plt.plot(fct1_mean.index, fct1_mean.values, label="prognoza ARMA(1,1)")
plt.fill_between(fct1_mean.index, fct1_ci.iloc[:,0], fct1_ci.iloc[:,1], alpha=0.2, label="95% CI")
plt.title("Prognozy ex-ante – ARMA(1,1)")
plt.xlabel("t"); plt.ylabel("X(t)"); plt.grid(True); plt.legend()
plt.tight_layout(); plt.show()

# b) ARIMA(1,2,1): order=(1,2,1) – jak w Twoim kodzie R
mod2 = sm.tsa.statespace.SARIMAX(Xts, order=(1,2,1), enforce_stationarity=False, enforce_invertibility=False)
res2 = mod2.fit(disp=False)
print("\n=== Model ręczny: ARIMA(1,2,1) ===")
print(res2.summary())

resid2 = res2.resid.dropna()

plt.figure(figsize=(9, 6))
plt.subplot(3,1,1); plt.plot(resid2); plt.title("Reszty ARIMA(1,2,1)"); plt.grid(True)
plt.subplot(3,1,2); plot_acf(resid2, ax=plt.gca(), lags=40)
plt.subplot(3,1,3); sm.qqplot(resid2, line="s", ax=plt.gca()); plt.title("QQ-plot reszt")
plt.tight_layout(); plt.show()

print("Shapiro-Wilk (reszty ARIMA(1,2,1)):", shapiro(resid2.values))
print("Ljung-Box p-values:\n", acorr_ljungbox(resid2, lags=[10, 20, 30], return_df=True)["lb_pvalue"])

fct2_res = res2.get_forecast(steps=H)
fct2_mean = fct2_res.predicted_mean
fct2_ci = fct2_res.conf_int()

plt.figure(figsize=(10, 4))
plt.plot(Xts, label="y")
plt.plot(fct2_mean.index, fct2_mean.values, label="prognoza ARIMA(1,2,1)")
plt.fill_between(fct2_mean.index, fct2_ci.iloc[:,0], fct2_ci.iloc[:,1], alpha=0.2, label="95% CI")
plt.title("Prognozy ex-ante – ARIMA(1,2,1)")
plt.xlabel("t"); plt.ylabel("X(t)"); plt.grid(True); plt.legend()
plt.tight_layout(); plt.show()

###############################################
# f) Dalsze eksperymenty:
# - podział na train/test, prognozy ex-post i miary błędu
###############################################

# Przykład prostego podziału ex-post:
test_h = 24 if len(Xts) > 60 else max(6, len(Xts)//5)
train = Xts.iloc[:-test_h]
test = Xts.iloc[-test_h:]

# Trenowanie auto_arima na train:
auto_tr = auto_arima(train, seasonal=(freq == 12), m=freq if freq == 12 else 1,
                     stepwise=True, trace=False, suppress_warnings=True)
pred = auto_tr.predict(n_periods=test_h)
pred = pd.Series(pred, index=test.index)

# Miary błędu:
mae = np.mean(np.abs(test - pred))
rmse = np.sqrt(np.mean((test - pred)**2))
mape = np.mean(np.abs((test - pred) / test)) * 100

print("\n=== Ex-post (train/test) ===")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)

plt.figure(figsize=(10,4))
plt.plot(train, label="train")
plt.plot(test, label="test")
plt.plot(pred, label="prognoza ex-post (auto_arima)")
plt.title("Ex-post: podział train/test i prognozy")
plt.xlabel("t"); plt.ylabel("X(t)"); plt.grid(True); plt.legend()
plt.tight_layout(); plt.show()


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [2]:
from pmdarima import auto_arima

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [3]:
from pmdarima.metrics import smape

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject